In [13]:
import pandas as pd
import numpy as np
import os

# Load the data
input_path = '../data/2026_meta_info.csv'
df = pd.read_csv(input_path)

# --- Configuration ---
strain_col = 'strain'
mic_cols = ['itc', 'pos', 'vrc'] 
# ---------------------

# 1. Initialize the master dataframe with PLINK ID columns
final_pheno = pd.DataFrame()
final_pheno['FID'] = df[strain_col]
final_pheno['IID'] = df[strain_col]


# 2. Loop through each drug and add the Log2 column
for drug in mic_cols:
    # Convert to numeric (handles 'NA' strings)
    numeric_mic = pd.to_numeric(df[drug], errors='coerce')
    
    # Calculate Log2 (using a mask to avoid log(0) errors if any MICs are 0)
    final_pheno[f'Log2_{drug}'] = np.log2(numeric_mic.replace(0, np.nan))

# 3. Handle Missing Data
# Replacing NaNs with -9 (Standard PLINK missing value indicator)
#final_pheno = final_pheno.fillna(int(0))
#final_pheno['FID'] = int(0)

# 4. Save to one single file
output_path = '../results/all_mic_phenotypes.txt'

# Ensure the results directory exists
os.makedirs('../results', exist_ok=True)

final_pheno.to_csv(output_path, sep='\t', index=False, quoting=3)

print(f"Successfully created {output_path} with traits: {', '.join(mic_cols)}")
print(final_pheno.head())

Successfully created ../results/all_mic_phenotypes.txt with traits: itc, pos, vrc
       FID      IID  Log2_itc  Log2_pos  Log2_vrc
0  V273-64  V273-64       0.0      -2.0       0.0
1  M039-56  M039-56       1.0      -1.0       1.0
2  M039-58  M039-58       0.0       1.0      -2.0
3  V080-81  V080-81       0.0      -2.0       0.0
4  V072-57  V072-57       5.0       1.0      -2.0


The below script converts a nwk file to matrix format 

In [4]:
from ete3 import Tree
import pandas as pd

t = Tree("../data/tree_data/asptu.raxml.tree", format=1)
samples = [leaf.name for leaf in t.iter_leaves()]
distances = pd.DataFrame(index=samples, columns=samples)

for i in samples:
    for j in samples:
        try:
            node_i = t.search_nodes(name=i)[0]
            node_j = t.search_nodes(name=j)[0]

            distances.loc[i, j] = t.get_distance(node_i, node_j)
        except IndexError:
            print(f"Missing node: {i} or {j}")
            distances.loc[i, j] = float('nan')

distances.to_csv("../results/species_distances.tsv", sep="\t")

Now we convert the distances to a PCA covariates format to be used in plink:

In [ ]:
import pandas as pd
from sklearn.decomposition import PCA

dist = pd.read_csv("../results/species_distances.tsv", sep="\t", index_col=0)
pca = PCA(n_components=10)
pcs = pca.fit_transform(dist)

# Convert to PLINK covariate format
# PLINK expects: FID, IID, PC1, PC2, ...
pca_df = pd.DataFrame(pcs, columns=[f'PC{i+1}' for i in range(10)])
pca_df.insert(0, 'IID', dist.index)
pca_df.insert(0, 'FID', dist.index)
# Save for PLINK
pca_df.to_csv("../results/pca_covariates.txt", sep="\t", index=False)

           FID        IID       PC1       PC2       PC3       PC4       PC5  \
0      V162-72    V162-72  2.657829 -0.540169  0.073317 -0.479580 -0.072816   
1    DTO241-E5  DTO241-E5  2.644933 -0.555922  0.061892 -0.477659 -0.077081   
2         1744       1744  2.563295 -0.650517 -0.008753 -0.461542 -0.102199   
3      M024-58    M024-58  4.710771 -0.119726  0.006414 -0.217026 -0.242606   
4          382        382  4.884730  0.091957  0.158240 -0.240122 -0.187921   
..         ...        ...       ...       ...       ...       ...       ...   
214        535        535 -1.595589 -0.892127 -0.050622 -0.064299 -0.005769   
215  ASFU-7528  ASFU-7528 -1.158123 -1.148851 -0.107179 -0.262114  0.061162   
216    V087-12    V087-12 -1.157870 -1.148535 -0.106948 -0.262150  0.061253   
217  ASFU-4617  ASFU-4617 -1.158519 -1.149339 -0.107543 -0.262052  0.061016   
218    V080-81    V080-81 -0.691394 -1.187741 -0.119092 -0.359658  0.067828   

          PC6       PC7       PC8       PC9      PC

Now we run the final part, first install plink and then run plink code:

In [ ]:
%%bash
VCF_FILE="/home/puntm/my_scratch_dir/2026_refbased/all_scaffolds.filteredSNPs.vcf.gz"

plink \
    --vcf "${VCF_FILE}" \
    --allow-extra-chr \
    --all-pheno \
    --linear \
    --allow-no-sex \
    --pheno "../results/all_mic_phenotypes.txt" \
    --covar "../results/pca_covariates.txt" \
    --out GWAS



bash: line 1: VCF_FILE: command not found
bash: line 4: plink: command not found


CalledProcessError: Command 'b'VCF_FILE ="$1"\n\nplink \\\n    --vcf "${VCF_FILE}" \\\n    --allow-extra-chr \\\n    --all-pheno \\\n    --linear \\\n    --no-sex \\\n    --assoc \\\n    --pheno "../results/all_mic_phenotypes.txt" \\\n    --covar "../results/pca_covariates.txt" \\\n    --out GWAS\n\n\n\n'' returned non-zero exit status 127.

Now we filter the only relevant info from thw GWAS:

# Keep the header and only rows that have 'ADD'
awk 'NR==1 || $5=="ADD" {print $0}' GWAS.Log2_itc.assoc.linear | gzip -9 > itc.out.gz
awk 'NR==1 || $5=="ADD" {print $0}' GWAS.Log2_pos.assoc.linear | gzip -9 > pos.out.gz
awk 'NR==1 || $5=="ADD" {print $0}' GWAS.Log2_vrc.assoc.linear | gzip -9 > vrc.out.gz
